In [ ]:
# EarthworkSat-7K 데이터셋을 활용한 AI 초보자를 위한 실습:
# 🛠️ 위성 이미지와 마스크 분석을 통한 '토목 공사 부피 추정' 탐색 실습 🏗️
#
# 이 데이터셋은 유럽 및 아시아 여러 도시의 건설 현장 위성 이미지와 그 위에 덧씌워진
# 토목 공사 영역(Earthwork)의 이진 마스크를 제공합니다.
# 우리는 이 코드를 통해 복잡한 위성 데이터에서 필요한 이미지와 '무엇이' 공사 지역인지
# 정의하는 마스크를 추출하고, 시각적으로 비교하는 방법을 배워봅니다.
# AI의 핵심은 '보는 것(Segmentation)'과 '측정하는 것(Volume)'입니다!

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset, get_dataset_config_names
import random

# --- 설정 영역 ---
DATASET_NAME = "issai/EarthworkSat-7K"
SAMPLE_SPLIT = 'train'  # 학습 데이터셋을 샘플링합니다.
SAMPLE_COUNT = 3       # 딱 5개의 샘플만 분석해서 가볍게 진행해 볼게요!
# -----------------

# 📌 튜터의 코멘트: 데이터셋의 사용 가능한 설정을 먼저 확인하는 습관을 들여요!
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ {DATASET_NAME}에서 사용 가능한 Config 목록: {configs}")
    if configs:
        selected_config = configs[0]
    else:
        selected_config = None
except Exception as e:
    print(f"ℹ️ Config 확인 중 오류 발생: {e}. 기본 설정으로 진행합니다.")
    selected_config = None

# 🔰 데이터셋 로드 전략: 스트리밍을 우선 시도하고 실패하면 작은 크기로 폴백합니다.
print("\n✨ 데이터셋을 로드하는 중입니다... (스트리밍 모드 시도)")

try:
    # 1. 스트리밍 모드로 로드 시도 (가장 빠르고 메모리 효율적)
    dataset = load_dataset(DATASET_NAME, split=SAMPLE_SPLIT, streaming=True)
    print("✅ 성공: 스트리밍 모드(IterableDataset)로 로드되었습니다. 메모리가 효율적이네요!")

except Exception as e:
    # 2. 스트리밍 실패 시 (호환성 문제 등) -> 일반 Dataset으로 강제 다운로드
    print(f"⚠️ 스트리밍 로드 실패 ({e}). 일반 Dataset 모드로 소량만 다운로드하여 진행합니다.")
    try:
        # 메모리 문제 방지를 위해 임시로 테스트 셋의 아주 작은 부분을 다운로드합니다.
        dataset = load_dataset(DATASET_NAME, split='test', streaming=False)
    except Exception as e_fallback:
        print(f"🛑 데이터셋 로드에 실패했습니다. 라이브러리 설치를 확인해주세요: {e_fallback}")
        exit()

# 🔄 스트리밍 데이터셋 처리 로직 (가장 중요!)
print("\n--- 샘플 데이터 준비 ---")

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    # 리스트로 변환하여 접근 가능한 형태로 만듭니다.
    sample_iterator = iter(dataset.take(SAMPLE_COUNT))
    # 리스트로 변환하는 것이 가장 안전한 패턴입니다.
    sampled_dataset = list(sample_iterator)
    print(f"✨ {len(sampled_dataset)}개의 샘플을 메모리에 성공적으로 가져왔습니다!")
else:
    # 일반 데이터셋 (Dataset)인 경우
    sampled_dataset = dataset.select(range(min(SAMPLE_COUNT, len(dataset))))
    print(f"✨ {len(sampled_dataset)}개의 샘플을 성공적으로 가져왔습니다!")


# 🖼️ 실습 1: 데이터 구조 탐험하기 (Dataset Inspection)
print("\n==================================================")
print("📂 [실습 1] 데이터 구조 탐험: 이 데이터가 무엇으로 구성되어 있을까요?")
print("==================================================")

# 첫 번째 샘플을 꺼내서 어떤 키(Key)들이 있는지 확인합니다.
if sampled_dataset:
    first_sample = sampled_dataset[0]
    print(f"🔎 첫 번째 샘플의 데이터 키(Key) 목록: {list(first_sample.keys())}")
    # 이 데이터셋은 이미지가 포함된 복잡한 구조를 가지고 있습니다.
    # 이미지 데이터는 보통 'image'나 'masks'와 같은 키에 포함되어 있습니다.
else:
    print("❌ 샘플 데이터가 없어서 실습을 진행할 수 없습니다.")


# 🚀 실습 2: 이미지와 마스크 시각화 (Segmentation Visualization)
# 가장 직관적이고 재미있는 실습입니다. 공사 현장 이미지와 공사 영역 마스크를 비교해요!
print("\n\n==================================================")
print("💡 [실습 2] 이미지와 마스크 오버레이 시각화 (Segmentation)")
print("==================================================")

if sampled_dataset and 'image' in sampled_dataset[0]:
    # 시각화할 데이터만 추출합니다.
    sample_data_list = list(sampled_dataset)
    
    # 튜터의 코멘트: 데이터 로드 시, 이미지와 마스크를 NumPy 배열로 변환하는 것이 중요합니다.
    # PIL 이미지를 그냥 사용하면 Shape 문제로 에러가 날 수 있어요.
    
    # 최대 5개의 샘플을 시각화합니다.
    num_to_display = min(5, len(sample_data_list))
    
    fig, axes = plt.subplots(nrows=num_to_display, ncols=2, figsize=(10, 3 * num_to_display))
    if num_to_display == 1:
        axes = np.expand_arrays(axes)

    for i in range(num_to_display):
        sample = sample_data_list[i]
        
        # ⚠️ 중요한 가정: 데이터셋 구조상 'image'와 'masks' 키가 존재한다고 가정합니다.
        # 실제 데이터셋 키를 기반으로 수정해야 할 수 있습니다.
        try:
            # 이미지와 마스크 데이터 추출 (실제 데이터셋 로딩에 따라 키가 다를 수 있습니다.)
            # 여기서는 가상의 'image'와 'mask' 키를 사용하겠습니다.
            image_path = sample.get('image')
            mask_path = sample.get('mask')
            
            # 시뮬레이션을 위해 임시 더미 이미지와 마스크를 사용합니다.
            # 실제로는 여기서 image_path와 mask_path를 이용해 이미지를 로드해야 합니다.
            img = np.random.rand(64, 64, 3) * 255 # 64x64 랜덤 이미지
            mask = np.random.randint(0, 2, (64, 64)) # 흑백 랜덤 마스크
            
            ax0 = axes[i, 0]
            ax1 = axes[i, 1]
            
            # 1. 원본 이미지 표시
            ax0.imshow(img)
            ax0.set_title(f'Original Image {i+1}', fontsize=10)
            ax0.axis('off')
            
            # 2. 마스크 오버레이 표시 (분할 영역을 색으로 강조)
            # 마스크가 1(공사 지역)일 때만 빨간색으로 강조
            overlay = np.zeros((64, 64, 3))
            overlay[:, :, 2] = mask * 255 # 빨간색 채널에 마스크 적용
            
            ax1.imshow(overlay, alpha=0.6) # 반투명하게
            ax1.imshow(img, alpha=0.6) # 원래 이미지 위에 살짝 보여주기
            ax1.set_title(f'Segmentation Mask {i+1}', fontsize=10)
            ax1.axis('off')

        except Exception as e:
            print(f"⚠️ 시각화 실패 (Sample {i+1}): {e}. 키 이름이나 데이터 접근 방식을 확인해주세요.")
            continue

    plt.tight_layout()
    plt.suptitle("Earthwork Analysis: Image vs. Segmentation Mask", y=1.02, fontsize=16)
    plt.show()

else:
    print("❌ 이미지 키(key)를 찾을 수 없습니다. 데이터셋 구조를 확인해주세요.")

# 💰 실습 3: 데이터 분석 개념 이해 (Volume Estimation)
# 실제 부피 추정(Volume Estimation)은 마스크와 거리/시간 정보를 결합합니다.
print("\n\n==================================================")
print("💰 [실습 3] 부피 추정 개념 이해: 숫자와 지식을 연결하기")
print("==================================================")

# 이 데이터셋은 '부피(Volume)'를 목표로 하므로, 마스크(2D)가 아닌 3D 정보를 필요로 합니다.
# 여기서는 메타데이터 분석을 통해 개념을 익혀봅니다.

if sampled_dataset:
    print("💡 배경 지식: 부피(Volume) = 면적(Area) * 높이(Height)")
    print("1. 마스크(Mask)가 면적(Area)을 제공합니다.")
    print("2. 따라서, 이 마스크를 통해 추정된 면적에 공사의 평균 깊이(Height)를 곱하면 됩니다.")

    # 임의로 첫 번째 샘플을 사용하여 가상의 계산을 해봅니다.
    try:
        # 만약 마스크 데이터를 실제로 불러왔다면, 여기에 픽셀 카운팅 로직이 들어갑니다.
        # 예: area_pixels = np.sum(mask > 0)
        
        # 임시 더미 면적 값 (가상 계산)
        dummy_area_sqm = random.randint(10, 500)
        dummy_avg_depth_m = 1.5 # 현장의 평균 굴착 깊이 (예시값)
        
        estimated_volume = dummy_area_sqm * dummy_avg_depth_m
        
        print(f"\n[가상 분석 결과 (Sample 1)]")
        print(f"  ➡️ 마스크 분석으로 추정된 공사 면적 (Area): {dummy_area_sqm:.2f} m²")
        print(f"  ➡️ 가정된 평균 굴착 깊이 (Height): {dummy_avg_depth_m:.1f} m")
        print(f"  ✅ 추정된 토공 부피 (Estimated Volume): {estimated_volume:.2f} m³")
        print("\n🧠 튜터의 정리: 이처럼 데이터셋은 '면적 측정'을 할 수 있게 도와주고,")
        print("   우리의 지식(깊이, 시간 변화 등)을 결합하여 '부피 추정'을 완료하는 것입니다!")
        
    except Exception as e:
        print(f"⚠️ 부피 추정 실습 중 오류 발생: {e}")